# Ablation — MLP + features géométriques

**Objectif :** Mesurer l'impact des features géométriques (vecteurs osseux, angles articulaires, normale de paume) sur le MLP baseline.

Entrée = 126 coordonnées normalisées + 156 features géométriques (60 vecteurs
osseux unitaires, 15 cos(angles) articulaires, 3 normale de paume × 2 mains) = 282 dimensions.
Résultat à comparer à la baseline (126 dimensions seules).

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → aucune fuite de données entre train et validation/test.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit 282 features = 126 coordonnées normalisées + 156 features géométriques.


In [2]:
dataset = load_dataset()
inp = normalized_inputs(dataset, with_geo=True)
X_train, y_train = torch.from_numpy(inp["train"]["features"]).float(), torch.from_numpy(inp["train"]["labels"])
X_val, y_val     = torch.from_numpy(inp["val"]["features"]).float(),   torch.from_numpy(inp["val"]["labels"])
X_test, y_test   = torch.from_numpy(inp["test"]["features"]).float(),  torch.from_numpy(inp["test"]["labels"])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 282) | val: (767, 282) | test: (767, 282)


## 2. Architecture

MLP résiduel identique à la baseline (mêmes blocs, même dropout), entrée 282.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = BaselineMLP(num_classes=57, input_size=282)
print(f"params: {count_params(model):,}")


params: 982,969


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 120 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=120, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "mlp_geo_curves.png",
                     title="Ablation — MLP + features géométriques")
print("best val acc:", trainer.best_val_acc)


  ep 1/120 train 3.6410/0.1550 val 2.4105/0.4798 (1.0s) best 0.4798


  ep 2/120 train 2.6974/0.3822 val 1.8524/0.6141 (1.3s) best 0.6141


  ep 3/120 train 2.2519/0.5054 val 1.5833/0.7301 (1.0s) best 0.7301


  ep 4/120 train 1.9983/0.5806 val 1.4723/0.7458 (1.3s) best 0.7458


  ep 5/120 train 1.8315/0.6376 val 1.4055/0.7744 (1.0s) best 0.7744


...

  ep 106/120 train 0.7834/0.9931 val 0.8018/0.9804 (1.3s) best 0.9844


  ep 107/120 train 0.7819/0.9934 val 0.8156/0.9726 (1.0s) best 0.9844


  ep 108/120 train 0.7830/0.9926 val 0.8075/0.9778 (1.3s) best 0.9844


  ep 109/120 train 0.7805/0.9938 val 0.8013/0.9817 (1.0s) best 0.9844
  early stop @ 109 (best val acc 0.9844)


best val acc: 0.984354628422425


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "mlp_geo_cm.png",
               title="Ablation — MLP + features géométriques — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_mlp_geo.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_mlp_geo.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.9804
macro F1: 0.9720 | weighted F1: 0.9803
macro precision: 0.9784 | macro recall: 0.9701


     class  precision    recall        f1  support
54   metro   0.000000  0.000000  0.000000        0
52  karhba   1.000000  0.500000  0.666667        2
8    labes   0.750000  0.750000  0.750000        4
10  mar7ba   0.750000  1.000000  0.857143        3
13     non   0.857143  0.923077  0.888889       13
17  siye7a   0.888889  0.888889  0.888889        9
22  telvza   1.000000  0.833333  0.909091        6
15   radio   0.952381  0.909091  0.930233       22


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
def forward_fn(model, raw, batch_size=512):
    # Perturbed raw landmarks -> wrist-normalized + geometric features (282-D)
    model.eval()
    from common.data import wrist_normalize, flatten, geometric_features
    feats = np.concatenate([flatten(wrist_normalize(raw)),
                            geometric_features(raw).reshape(len(raw), -1)], axis=1)
    outs = []
    with torch.no_grad():
        for i in range(0, len(feats), batch_size):
            x = torch.from_numpy(feats[i:i + batch_size]).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.8229 (drop=+0.1575)
  noise_0.02: acc=0.7272 (drop=+0.2532)
  noise_0.05: acc=0.4162 (drop=+0.5643)


  rotation_5: acc=0.9791 (drop=+0.0013)


  rotation_10: acc=0.9531 (drop=+0.0274)
  rotation_20: acc=0.8540 (drop=+0.1265)
  translation_0.05: acc=0.9804 (drop=+0.0000)
  translation_0.10: acc=0.9804 (drop=+0.0000)


  scale_0.90: acc=0.9804 (drop=+0.0000)


  scale_1.10: acc=0.9804 (drop=+0.0000)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "mlp_geo")
fp16 = ex.quantize_fp16(onnx_path, "mlp_geo")
int8 = ex.quantize_int8(onnx_path, "mlp_geo")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("mlp_geo", metrics, extra=extra)

torch_path = ex.save_torch(model, "mlp_geo")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.323 ms/sample


  fp32: size=3.77 MB latency=0.083 ms acc=0.9804
  fp16: size=1.89 MB latency=0.092 ms acc=0.9804
  int8: size=0.99 MB latency=0.075 ms acc=0.9804
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_mlp_geo.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\mlp_geo.pth (3.79 MB)


## 7. Conclusions

Comparaison directe baseline vs baseline+geo : si le gain est significatif, les
features géométriques encodent de l'information que le MLP ne peut pas extraire seul des
coordonnées (invariance d'échelle/translation des angles et vecteurs unitaires).
